In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week6-lesson-4"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
master('yarn'). \
getOrCreate()


### enableHiveSupport(). if we dont specify this, spark will create a temporary in memory metastore

In [3]:
# from pyspark.sql import SparkSession

# spark = SparkSession. \
# builder. \
# appName("week6-lesson-4"). \
# config('spark.ui.port','0'). \
# config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
# enableHiveSupport(). \ ##----  if we dont specify this, spark will create a temporary in memory metastore
# master('local[*]'). \  ##--- submits job to local machine, not the cluster
# getOrCreate()

In [4]:
print(spark)

### spark2 = spark.newSession()

In [5]:
spark2 = spark.newSession()  ## create an isolated child session that shares the SparkContext but has its own 
# session-local configs/temp views (rarely needed, but good to know)

In [6]:
print(spark2)

In [7]:
df_orders = spark.read \
.format("csv") \
.option("header","true") \
.option("inferSchema","true")\
.load("/public/trendytech/orders_wh")

In [22]:
df_orders.createOrReplaceTempView("df_view")

In [23]:
spark.sql("select * from df_view limit 5")

order_id,order_date,customer_id,order_status
1,2013-07-25 00:00:...,11599,CLOSED
2,2013-07-25 00:00:...,256,PENDING_PAYMENT
3,2013-07-25 00:00:...,12111,COMPLETE
4,2013-07-25 00:00:...,8827,CLOSED
5,2013-07-25 00:00:...,11318,COMPLETE


In [ ]:
spark2.sql("select * from df_view limit 5") ## will error out as the session is different from where it was created

In [24]:
spark.sql("show tables")

database,tableName,isTemporary
,df_view,true


In [29]:
spark2.sql("show tables")  ## view not visible

database,tableName,isTemporary


In [13]:
# temp views are not visible across different spark sessions

### createOrReplaceGlobalTempView  -  global_temp

In [28]:
df_orders.createOrReplaceGlobalTempView("df_view1")  ## GlobalTempView are visible across spark sessions. use global_temp as DB name

In [26]:
spark.sql("select * from global_temp.df_view1 limit 5")

order_id,order_date,customer_id,order_status
1,2013-07-25 00:00:...,11599,CLOSED
2,2013-07-25 00:00:...,256,PENDING_PAYMENT
3,2013-07-25 00:00:...,12111,COMPLETE
4,2013-07-25 00:00:...,8827,CLOSED
5,2013-07-25 00:00:...,11318,COMPLETE


In [27]:
spark2.sql("select * from global_temp.df_view1 limit 5")

order_id,order_date,customer_id,order_status
1,2013-07-25 00:00:...,11599,CLOSED
2,2013-07-25 00:00:...,256,PENDING_PAYMENT
3,2013-07-25 00:00:...,12111,COMPLETE
4,2013-07-25 00:00:...,8827,CLOSED
5,2013-07-25 00:00:...,11318,COMPLETE
